# Creative Performance Score Evolution

This notebook analyzes how a unified performance metric (`perf_score` proxy, typically driven by ROAS and Conv. Rate) evolves over time for selected creatives. It includes visual charting of the performance decay and a simple forecasting model (e.g., using Linear Regression or Exponential Smoothing) to predict future performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set plot styles
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook", font_scale=1.1)

# Load datasets
try:
    daily_stats = pd.read_csv("../data/creative_daily_country_os_stats.csv", parse_dates=["date"])
    creatives = pd.read_csv("../data/creative_summary.csv")
    print(f"Loaded {len(daily_stats)} daily records and {len(creatives)} creative summaries.")
except FileNotFoundError:
    print("Run this from the notebooks/ directory or adjust the path.")

In [ ]:
# Aggregate daily performance at the creative level
daily_creative = (
    daily_stats.groupby(["creative_id", "date", "days_since_launch"])
    .agg(
        {
            "impressions": "sum",
            "clicks": "sum",
            "conversions": "sum",
            "spend_usd": "sum",
            "revenue_usd": "sum",
        }
    )
    .reset_index()
)

# Calculate daily proxy metrics
daily_creative["daily_roas"] = daily_creative["revenue_usd"] / daily_creative["spend_usd"].replace(
    0, np.nan
)
daily_creative["daily_ctr"] = daily_creative["clicks"] / daily_creative["impressions"].replace(
    0, np.nan
)
daily_creative["daily_cvr"] = daily_creative["conversions"] / daily_creative["clicks"].replace(
    0, np.nan
)

# To mimic a unified 'perf_score' locally, we can take a scaled ROAS or standard Revenue per Mille (RPM)
daily_creative["daily_rpm"] = (
    daily_creative["revenue_usd"] / daily_creative["impressions"].replace(0, np.nan)
) * 1000

# Sort by creative and date to ensure rolling window makes chronological sense
daily_creative = daily_creative.sort_values(["creative_id", "date"])

# Let's define the proxy for our perf score evolution as a smoothed ROAS
daily_creative["smoothed_roas"] = daily_creative.groupby("creative_id")["daily_roas"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# Merge with creative summary to know the status
daily_creative = daily_creative.merge(
    creatives[["creative_id", "creative_status", "perf_score"]], on="creative_id", how="left"
)

daily_creative.head()

In [ ]:
# Select a few creatives to plot
# Get one from each primary status type
example_creatives = []
for status in ["top_performer", "stable", "fatigued"]:
    sample = creatives[creatives["creative_status"] == status]["creative_id"].dropna().head(2)
    example_creatives.extend(sample.tolist())

plot_df = daily_creative[daily_creative["creative_id"].isin(example_creatives)]

# Plot the evolution of daily ROAS smoothed
plt.figure(figsize=(14, 8))
sns.lineplot(
    data=plot_df,
    x="days_since_launch",
    y="smoothed_roas",
    hue="creative_id",
    style="creative_status",
    linewidth=2,
    markers=True,
    dashes=False,
)

plt.title(
    "Performance Evolution (Smoothed ROAS) over Days Since Launch", fontsize=16, fontweight="bold"
)
plt.xlabel("Days Since Launch", fontsize=12)
plt.ylabel("Smoothed ROAS", fontsize=12)
plt.axhline(1.0, color="red", linestyle="--", alpha=0.5, label="Breakeven ROAS (1.0)")
plt.legend(title="Creative info", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Define a simple forecasting pipeline using ARIMA models with automatic order selection
import itertools
import warnings

from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")


def optimize_arima(y):
    """Grid search to find the optimal ARIMA(p,d,q) order based on AIC."""
    best_aic = np.inf
    best_order = None
    best_model_fit = None

    # Test basic combinations of p, d, q
    p_values = range(0, 3)
    d_values = range(0, 2)
    q_values = range(0, 3)

    for param in itertools.product(p_values, d_values, q_values):
        try:
            model = ARIMA(y, order=param)
            model_fit = model.fit()
            if model_fit.aic < best_aic:
                best_aic = model_fit.aic
                best_order = param
                best_model_fit = model_fit
        except Exception:
            continue

    return best_order, best_model_fit


def plot_forecast_for_creative(creative_df, forecast_days=14):
    """
    Train an ARIMA model (automatically selecting order) on the creative's
    perf score (smoothed_roas) to forecast its remaining lifespan.
    """
    creative_df = creative_df.sort_values("days_since_launch").dropna(subset=["smoothed_roas"])

    # ARIMA typically needs a few more data points than a simple polynomial fit
    if len(creative_df) < 10:
        print(
            f"Not enough data for creative {creative_df['creative_id'].iloc[0]} to fit an ARIMA model."
        )
        return

    # Extract inputs
    X = creative_df["days_since_launch"].values
    y = creative_df["smoothed_roas"].values

    print("Finding best ARIMA order...")
    best_order, model_fit = optimize_arima(y)

    if model_fit is None:
        print("ARIMA fitting failed across all tested orders.")
        return

    print(f"Optimal ARIMA order found: {best_order} (AIC: {model_fit.aic:.2f})")

    # Extrapolate over future days
    last_day = X[-1]
    future_days = np.arange(last_day + 1, last_day + 1 + forecast_days)

    # Predict over the training data and forecast out
    model_predictions = model_fit.predict(start=0, end=len(y) - 1)
    future_roas = model_fit.forecast(steps=forecast_days)

    # Plotting
    plt.figure(figsize=(10, 6))

    # Historical Data and In-sample fit
    plt.scatter(X, y, color="blue", alpha=0.6, label="Historical Smoothed ROAS")
    plt.plot(
        X,
        model_predictions,
        color="darkblue",
        linestyle="-",
        linewidth=2,
        label=f"ARIMA{best_order} Fit",
    )

    # Forecast Data
    plt.plot(
        future_days,
        future_roas,
        color="orange",
        linestyle="--",
        linewidth=2,
        label=f"{forecast_days}-Day Forecast",
    )

    creative_base_status = creative_df["creative_status"].iloc[0]
    creative_id = creative_df["creative_id"].iloc[0]

    plt.title(
        f"Performance Forecast for Creative #{creative_id} ({creative_base_status}) - Auto ARIMA",
        fontweight="bold",
    )
    plt.xlabel("Days Since Launch")
    plt.ylabel("Smoothed ROAS")

    # Threshold for poor performance
    plt.axhline(0.8, color="red", linestyle=":", label="Threshold to Pause (0.8)")

    # Shade the forecast zone
    plt.axvspan(last_day, last_day + forecast_days, color="orange", alpha=0.1)

    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.show()


# Pick one fatigued creative and forecast its future performance
target_creative = creatives[creatives["creative_status"] == "fatigued"]["creative_id"].iloc[0]
forecast_df = daily_creative[daily_creative["creative_id"] == target_creative]

# Run the automated forecast
plot_forecast_for_creative(forecast_df, forecast_days=10)